# InmoCore V3 Enterprise
### Notebook de demostración

Este notebook recorre las características principales del proyecto:
1. **Multi-tenancy**: aislamiento de datos por agencia.
2. **Alertas automáticas** por precio anómalo.
3. **Publicación de una propiedad** y búsqueda semántica con ChromaDB.
4. **Búsqueda geoespacial** (`$nearSphere`) alrededor de la agencia.
5. **Reporte agregado** (Aggregation Pipeline) de propiedades por tipo.

Antes de correr este notebook, ejecutá `python setup_db.py` y `python seed.py` para tener datos de prueba cargados.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from dao import AdminDAO, InmoCoreDAO, StorageDAO, VectorDAO

# 1. Usar AdminDAO para ver las agencias (tenants) registradas
admin_dao = AdminDAO()
agencias = list(admin_dao.col_agencias.find())
admin_dao.cerrar_conexion()

if not agencias:
    raise Exception("No hay agencias. Ejecutá seed.py primero.")

print("Agencias Disponibles (Tenants):")
for a in agencias:
    print(f"- {a['nombre']} (ID: {a['_id']})")

# Elegimos la primera agencia para instanciar el DAO operativo
agencia_activa = agencias[0]
dao = InmoCoreDAO(agencia_id=str(agencia_activa['_id']))
print(f"\nDAO instanciado para: {agencia_activa['nombre']} — Asegurando aislamiento multi-tenant.")


## 1. Verificación de Alertas Automáticas
Al insertar una propiedad en Venta con un precio menor a 10.000 USD, el DAO genera automáticamente una alerta de "Precio Anómalo".

In [ ]:
alertas = dao.listar_alertas()
print(f"Alertas Activas: {len(alertas)}")
for a in alertas:
    print(f"[! {a.tipo_alerta} !] {a.mensaje} (Fecha: {a.fecha_alerta})")


## 2. Publicar una propiedad e indexarla semánticamente
Publicamos una propiedad nueva, la guardamos en MongoDB y la indexamos en ChromaDB para poder encontrarla luego por lenguaje natural.

In [ ]:
from db_models import Propiedad

nueva = Propiedad(
    agencia_id=str(agencia_activa['_id']),
    titulo="Departamento luminoso con balcón en Palermo",
    descripcion="Departamento moderno, muy luminoso, con balcón al frente y cochera doble. Ideal para parejas jóvenes.",
    tipo="Departamento",
    operacion="Venta",
    precio_usd=145000.0,
    superficie_m2=58.0,
    habitaciones=2,
    ubicacion={"type": "Point", "coordinates": [-58.42, -34.58]}
)
nueva_id = dao.insertar_propiedad(nueva)

vector_dao = VectorDAO()
vector_dao.indexar_descripcion(nueva_id, nueva.descripcion)

print(f"Propiedad publicada con id: {nueva_id}")


## 3. Búsqueda Semántica (ChromaDB)
Buscamos propiedades por similitud de significado, no por coincidencia exacta de texto.

In [ ]:
resultados = vector_dao.buscar_similitud("depto con balcón y cochera para pareja", n_results=3)

for i, propiedad_id in enumerate(resultados['ids'][0]):
    propiedad = dao.obtener_propiedad(propiedad_id)
    if not propiedad:
        continue
    distancia = resultados['distances'][0][i]
    similitud = max(0, int((1 - distancia) * 100))
    print(f"{propiedad.titulo} — ${propiedad.precio_usd:,.0f} — Similitud: {similitud}%")


## 4. Búsqueda Geoespacial (`$nearSphere`)
Buscamos propiedades de la agencia dentro de un radio de 3 km alrededor de su ubicación.

In [ ]:
lon, lat = agencia_activa["ubicacion"]["coordinates"]
cercanas = dao.buscar_propiedades_cerca_de(lat=lat, lon=lon, radio_km=3)

print(f"Propiedades encontradas dentro de 3 km: {len(cercanas)}")
for p in cercanas[:5]:
    print(f"- {p.titulo} ({p.tipo}, {p.operacion}) — ${p.precio_usd:,.0f}")


## 5. Reporte Agregado (Aggregation Pipeline)
Cantidad de propiedades y precio promedio, agrupados por tipo.

In [ ]:
reporte = dao.reporte_propiedades_por_tipo()
df_reporte = pd.DataFrame(reporte)

if not df_reporte.empty:
    df_reporte.rename(columns={"_id": "Tipo", "cantidad": "Cantidad", "precio_promedio_usd": "Precio Promedio USD"}, inplace=True)
    display(df_reporte)

    plt.figure(figsize=(10, 5))
    plt.bar(df_reporte["Tipo"], df_reporte["Cantidad"], color="teal")
    plt.title(f"Cantidad de Propiedades por Tipo — {agencia_activa['nombre']}")
    plt.xlabel("Tipo de Propiedad")
    plt.ylabel("Cantidad")
    plt.tight_layout()
    plt.show()

dao.cerrar_conexion()
